In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch
from torchvision import datasets
from efficientnet_pytorch import EfficientNet  
import torch.nn as nn
import torch.optim as optim
from collections import Counter
from torch.utils.data import WeightedRandomSampler, DataLoader
import numpy as np
from pathlib import Path

class AlbumentationsWrapper:
    def __init__(self, transform):
        self.transform = transform

    def __call__(self, img):
        img_np = np.array(img)
        augmented = self.transform(image=img_np)
        return augmented['image']
    
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.alpha is not None:
            focal_loss = self.alpha[targets] * focal_loss
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss
    
def train():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}\n")

    # TRANSFORMS  
    train_aug = A.Compose([
        A.Resize(224, 224),
        A.Rotate(limit=30, p=0.7),
        A.VerticalFlip(p=0.4),
        A.HorizontalFlip(p=0.5),  
        A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
        A.GaussianBlur(blur_limit=3, p=0.3),
        A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    
    val_aug = A.Compose([
        A.Resize(224, 224),
        A.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
        ToTensorV2(),
    ])

    # LOAD DATA
    train_data = datasets.ImageFolder(
        "/content/my_data/processed/train", 
        transform=AlbumentationsWrapper(train_aug)
    )
    val_data = datasets.ImageFolder(
        "/content/my_data/processed/val", 
        transform=AlbumentationsWrapper(val_aug)
    )
    
    print(f"Train samples: {len(train_data)}")
    print(f"Val samples: {len(val_data)}")
    print(f"Classes: {train_data.classes}\n")

    # CLASS BALANCING
    class_counts = Counter(train_data.targets)
    print("Class distribution:")
    for class_idx, count in sorted(class_counts.items()):
        print(f"  {train_data.classes[class_idx]}: {count} images")
    
    weight_per_class = [1.0/class_counts[i] for i in range(len(class_counts))]

    print(f"Enhanced weights: {weight_per_class}")

    sample_weight = [weight_per_class[label] for label in train_data.targets]
    sampler = WeightedRandomSampler(
        weights=sample_weight,
        num_samples=len(sample_weight),
        replacement=True
    )
    
    # DATA LOADERS
    train_loader = DataLoader(
        train_data,
        batch_size=32,
        sampler=sampler,
        num_workers=2
    )
    val_loader = DataLoader(
        val_data,
        batch_size=32,
        shuffle=False,
        num_workers=2
    )

    # MODEL SETUP    
    print("\nLoading EfficientNet-B0...")
    model = EfficientNet.from_pretrained('efficientnet-b0')
    
    # Freeze all layers
    for param in model.parameters():
        param.requires_grad = False

    # Replace final layer
    num_classes = len(train_data.classes)
    model._fc = nn.Linear(model._fc.in_features, num_classes)
    model = model.to(device)
    print(f"Model loaded. Training {num_classes} classes.\n")

    #Focal loss 
    criterion = FocalLoss(gamma=2.0)
    print("Using FocalLoss (gamma=2.0) for hard example focus.\\n")

    # PHASE 1: TRAIN HEAD ONLY 
    print("="*60)
    print("PHASE 1: Training head only (5 epochs)")
    print("="*60)
    
    optimizer = optim.Adam(model._fc.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    best_val_acc = 0.0

    for epoch in range(5):
        # ─── Training ───
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward
            outputs = model(images)
            loss = criterion(outputs, labels)

            # Backward
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Statistics
            train_loss += loss.item()  
            _, predicted = torch.max(outputs.data, dim=1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = 100 * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)

        # ─── Validation ───
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, dim=1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        
        print(f"\nEpoch {epoch+1}/3")
        print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

        if val_acc > best_val_acc:  
            best_val_acc = val_acc
            Path('./models').mkdir(exist_ok=True)
            torch.save(model.state_dict(), './models/best_model.pth')
            print(f"  ✅ Saved (best: {best_val_acc:.2f}%)")

    # PHASE 2: FINE-TUNE ALL LAYERS    
    print("\n" + "="*60)
    print("PHASE 2: Fine-tuning all layers (15 epochs)")
    print("="*60)
    
    # Unfreeze all
    for param in model.parameters():
        param.requires_grad = True
    
    # Small learning rate for fine-tuning
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(25):
        # ─── Training ───
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = torch.max(outputs.data, dim=1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        train_acc = 100 * train_correct / train_total
        avg_train_loss = train_loss / len(train_loader)

        # ─── Validation ───
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, dim=1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
        
        val_acc = 100 * val_correct / val_total
        avg_val_loss = val_loss / len(val_loader)
        
        print(f"\nEpoch {epoch+1}/15")
        print(f"  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss:   {avg_val_loss:.4f} | Val Acc:   {val_acc:.2f}%")

        if val_acc > best_val_acc:  
            best_val_acc = val_acc
            torch.save(model.state_dict(), './models/best_model.pth')
            print(f" Saved (best: {best_val_acc:.2f}%)")

    print("\n" + "="*60)
    print(f"Training complete! Best accuracy: {best_val_acc:.2f}%")
    print("="*60)

if __name__ == "__main__":
    train()

Using device: cuda

Train samples: 15840
Val samples: 3392
Classes: ['10', '100', '1000', '20', '5', '50', '500']

Class distribution:
  10: 2900 images
  100: 2581 images
  1000: 2232 images
  20: 2155 images
  5: 1400 images
  50: 2348 images
  500: 2224 images
Enhanced weights: [0.0003448275862068965, 0.0003874467260751647, 0.00044802867383512545, 0.0004640371229698376, 0.0007142857142857143, 0.00042589437819420784, 0.0004496402877697842]

Loading EfficientNet-B0...


/tmp/ipykernel_1213/3869072210.py:54: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=4, max_height=32, max_width=32, p=0.3),


Loaded pretrained weights for efficientnet-b0
Model loaded. Training 7 classes.

Using FocalLoss (gamma=2.0) for hard example focus.\n
PHASE 1: Training head only (5 epochs)


In [ ]:
!pip install efficientnet_pytorch

In [ ]:
from google.colab import files # type: ignore
files.download('models/best_model.pth')

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!ls /content/drive/MyDrive/data.zip

In [ ]:
!unzip /content/drive/MyDrive/data.zip -d /content/my_data